# 03 · Agua superficial con NDWI y MNDWI

**Objetivo:** Detectar cuerpos de agua y comparar dos índices espectrales.

**Datos:** Sentinel-2 SR: B3, B8 y B11.

**Relevancia para política ambiental y social:** Útil para humedales, seguridad hídrica y cambios en ríos o lagunas.

**Limitaciones:** Sombras y superficies oscuras pueden producir falsos positivos.


In [ ]:
# Instalar dependencias en Google Colab
!pip -q install earthengine-api geemap

import ee
import geemap
import datetime

ee.Authenticate()
ee.Initialize(project="TU_PROYECTO_GEE")

# Área de estudio de ejemplo: entorno de Chachapoyas, Amazonas, Perú
# Reemplázala por un polígono, activo de Earth Engine o coordenadas propias.
aoi = ee.Geometry.Point([-77.87, -6.23]).buffer(30000)

Map = geemap.Map()
Map.centerObject(aoi, 9)


In [ ]:
def mask_s2_sr(image):
    scl = image.select("SCL")
    clear = (
        scl.neq(3)   # sombra
        .And(scl.neq(8))  # nube media
        .And(scl.neq(9))  # nube alta
        .And(scl.neq(10)) # cirrus
        .And(scl.neq(11)) # nieve/hielo
    )
    return image.updateMask(clear).divide(10000).copyProperties(
        image, ["system:time_start"]
    )

def s2_composite(start, end):
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
        .map(mask_s2_sr)
        .median()
        .clip(aoi)
    )


In [ ]:
image = s2_composite("2026-01-01", "2026-07-29")
ndwi = image.normalizedDifference(["B3","B8"]).rename("NDWI")
mndwi = image.normalizedDifference(["B3","B11"]).rename("MNDWI")
water = mndwi.gt(0.1).selfMask()

Map.addLayer(ndwi, {"min":-0.5,"max":0.7,"palette":["brown","white","blue"]}, "NDWI", False)
Map.addLayer(mndwi, {"min":-0.5,"max":0.7,"palette":["brown","white","blue"]}, "MNDWI")
Map.addLayer(water, {"palette":["cyan"]}, "Agua probable")
Map
